# Avaliação do modelo final

Este notebook avalia o modelo campeão **já salvo** (`models/churn_pipeline.pkl`) no conjunto de teste e gera todos os
números apresentados no Model Card (`docs/model_card.md`):

- métricas no conjunto de teste e matriz de confusão (seção 5.1);
- sensibilidade ao limiar de decisão (seção 5.4);
- variáveis mais influentes (seção 5.5);
- calibração das probabilidades (limitação 3);
- desempenho por segmento (limitação 5);
- possíveis vieses e análise de equidade por grupo (seção 7);
- impacto de negócio, usado no ML Canvas e na seção 5.6.

O conjunto de teste é recriado com o mesmo split e a mesma seed de `src/train_final_model.py`, então os 1.409 clientes
avaliados aqui são exatamente os que o modelo não viu no treino.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split

from pipeline import LIMIAR_CHURN
from predict import carregar_modelo
from preprocessing import carregar_dados, preparar_features, tratar_total_charges
from train_final_model import SEED

df = tratar_total_charges(carregar_dados('../data/Telco-Customer-Churn.csv'))
df_model = preparar_features(df)
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

modelo = carregar_modelo('../models/churn_pipeline.pkl')
prob = pd.Series(modelo.predict_proba(X_test)[:, 1], index=X_test.index)
pred = (prob >= LIMIAR_CHURN).astype(int)

print(f'Treino: {len(X_train)} clientes | Teste: {len(X_test)} clientes '
      f'({(y_test == 0).sum()} não-churn, {(y_test == 1).sum()} churn) | Limiar: {LIMIAR_CHURN}')

Treino: 5634 clientes | Teste: 1409 clientes (1035 não-churn, 374 churn) | Limiar: 0.5


## 5.1 Métricas no conjunto de teste e matriz de confusão

Recall, precision, F1 e acurácia usam o limiar de decisão. ROC-AUC e PR-AUC avaliam a ordenação dos clientes por risco,
independentemente do limiar. O Brier score mede o erro quadrático médio das probabilidades (quanto menor, melhor).

In [2]:
metricas_teste = pd.Series({
    'Recall (churn)': recall_score(y_test, pred),
    'Precision (churn)': precision_score(y_test, pred),
    'F1-score (churn)': f1_score(y_test, pred),
    'Acurácia': accuracy_score(y_test, pred),
    'ROC-AUC': roc_auc_score(y_test, prob),
    'PR-AUC (Average Precision)': average_precision_score(y_test, prob),
    'Brier score': brier_score_loss(y_test, prob),
}).round(3)
metricas_teste.to_frame('valor')

,valor
Recall (churn),0.794
Precision (churn),0.518
F1-score (churn),0.627
Acurácia,0.749
ROC-AUC,0.844
PR-AUC (Average Precision),0.664
Brier score,0.166


In [3]:
vn, fp, fn, vp = confusion_matrix(y_test, pred).ravel()
pd.DataFrame(
    [[f'{vn} (VN)', f'{fp} (FP)'], [f'{fn} (FN)', f'{vp} (VP)']],
    index=['Real: não churn', 'Real: churn'],
    columns=['Previsto: não churn', 'Previsto: churn'],
)

,Previsto: não churn,Previsto: churn
Real: não churn,759 (VN),276 (FP)
Real: churn,77 (FN),297 (VP)


## 5.4 Sensibilidade ao limiar de decisão

Como o limiar muda o equilíbrio entre recall e precision, e quantos clientes seriam sinalizados para o time de retenção.

In [4]:
linhas = []
for limiar in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_limiar = (prob >= limiar).astype(int)
    linhas.append({
        'limiar': limiar,
        'recall': recall_score(y_test, pred_limiar),
        'precision': precision_score(y_test, pred_limiar),
        'f1': f1_score(y_test, pred_limiar),
        '% da base sinalizada': 100 * pred_limiar.mean(),
    })

pd.DataFrame(linhas).set_index('limiar').round(2)

,recall,precision,f1,% da base sinalizada
limiar,,,,
0.3,0.93,0.42,0.58,58.06
0.4,0.86,0.46,0.60,49.68
0.5,0.79,0.52,0.63,40.67
0.6,0.72,0.56,0.63,33.71
0.7,0.59,0.63,0.61,24.77


## 5.5 Variáveis mais influentes

Coeficientes da regressão logística depois do pré-processamento (numéricas padronizadas, categóricas em One-Hot).
Coeficiente positivo aumenta o risco de churn; negativo reduz.

In [5]:
nomes = modelo.named_steps['pre'].get_feature_names_out()
coeficientes = pd.Series(modelo.named_steps['modelo'].coef_[0], index=nomes)
coeficientes.index = (coeficientes.index
                      .str.replace(r'^(num|cat|bin)__', '', regex=True)
                      .str.replace(r'^(\w+?)_(?=[A-Z0-9])', r'\1 = ', regex=True))

top10 = coeficientes.reindex(coeficientes.abs().sort_values(ascending=False).index).head(10)
top10.round(2).to_frame('coeficiente')

,coeficiente
tenure,-1.35
Contract = Two year,-0.82
TotalCharges,0.65
Contract = Month-to-month,0.64
InternetService = DSL,-0.32
OnlineSecurity = Yes,-0.28
InternetService = Fiber optic,0.28
PhoneService = Yes,-0.27
SeniorCitizen,0.26
PaymentMethod = Electronic check,0.24


In [6]:
# gênero: compara os coeficientes das duas categorias
coeficientes[coeficientes.index.str.startswith('gender')].round(3).to_frame('coeficiente')

,coeficiente
gender = Female,-0.076
gender = Male,-0.074


## Limitação 3: calibração das probabilidades

Com `class_weight='balanced'`, o modelo é treinado como se as classes tivessem o mesmo tamanho, o que infla as
probabilidades da classe churn. Comparamos a taxa de churn real com a média das probabilidades previstas e com a
fração de clientes sinalizados.

In [7]:
pd.Series({
    'Churn real no teste': y_test.mean(),
    'Probabilidade média prevista': prob.mean(),
    'Fração sinalizada (limiar 0,5)': pred.mean(),
}).round(3).to_frame('valor')

,valor
Churn real no teste,0.265
Probabilidade média prevista,0.414
"Fração sinalizada (limiar 0,5)",0.407


## Métricas por grupo

Função usada nas duas próximas seções: calcula, para cada grupo de clientes do conjunto de teste, o tamanho, a taxa
real de churn, a fração sinalizada pelo modelo, recall, precision, taxa de falsos positivos (FPR: clientes que não
cancelariam, mas foram sinalizados) e ROC-AUC.

In [8]:
def metricas_por_grupo(grupos: dict) -> pd.DataFrame:
    linhas = []
    for nome, mascara in grupos.items():
        y_g, prob_g, pred_g = y_test[mascara], prob[mascara], pred[mascara]
        linhas.append({
            'grupo': nome,
            'n': len(y_g),
            'churn real (n)': int(y_g.sum()),
            'churn real (%)': 100 * y_g.mean(),
            '% sinalizado': 100 * pred_g.mean(),
            'recall': recall_score(y_g, pred_g, zero_division=0),
            'precision': precision_score(y_g, pred_g, zero_division=0),
            'FPR': pred_g[y_g == 0].mean(),
            'ROC-AUC': roc_auc_score(y_g, prob_g) if y_g.nunique() == 2 else float('nan'),
        })
    return pd.DataFrame(linhas).set_index('grupo').round(2)

## Limitação 5: desempenho por segmento

Segmentos com pouco churn (contratos longos, clientes sem internet, clientes antigos), onde os cancelamentos que
acontecem tendem a passar despercebidos.

In [9]:
metricas_por_grupo({
    'Contrato mensal': X_test['Contract'] == 'Month-to-month',
    'Contrato de 1 ano': X_test['Contract'] == 'One year',
    'Contrato de 2 anos': X_test['Contract'] == 'Two year',
    'Internet: fibra': X_test['InternetService'] == 'Fiber optic',
    'Internet: DSL': X_test['InternetService'] == 'DSL',
    'Sem internet': X_test['InternetService'] == 'No',
    'Até 12 meses de casa': X_test['tenure'] <= 12,
    'Mais de 4 anos de casa': X_test['tenure'] > 48,
})

,n,churn real (n),churn real (%),% sinalizado,recall,precision,FPR,ROC-AUC
grupo,,,,,,,,
Contrato mensal,773,335,43.34,71.28,0.87,0.53,0.60,0.74
Contrato de 1 ano,290,27,9.31,7.24,0.26,0.33,0.05,0.74
Contrato de 2 anos,346,12,3.47,0.29,0.00,0.00,0.00,0.62
Internet: fibra,617,254,41.17,66.77,0.90,0.55,0.51,0.80
Internet: DSL,492,95,19.31,32.32,0.73,0.43,0.23,0.81
Sem internet,300,25,8.33,0.67,0.00,0.00,0.01,0.82
Até 12 meses de casa,433,205,47.34,69.98,0.89,0.60,0.53,0.78
Mais de 4 anos de casa,451,44,9.76,9.53,0.41,0.42,0.06,0.81


## 7. Possíveis vieses e análise de equidade

Mesmas métricas para grupos definidos por atributos sensíveis ou que podem funcionar como proxy de perfil
socioeconômico (método de pagamento).

In [10]:
pagamento_automatico = X_test['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)'])

metricas_por_grupo({
    'Feminino': X_test['gender'] == 'Female',
    'Masculino': X_test['gender'] == 'Male',
    'Não idoso': X_test['SeniorCitizen'] == 0,
    'Idoso': X_test['SeniorCitizen'] == 1,
    'Sem parceiro(a)': X_test['Partner'] == 'No',
    'Com parceiro(a)': X_test['Partner'] == 'Yes',
    'Sem dependentes': X_test['Dependents'] == 'No',
    'Com dependentes': X_test['Dependents'] == 'Yes',
    'Pagamento: cheque eletrônico': X_test['PaymentMethod'] == 'Electronic check',
    'Pagamento: automático (cartão/banco)': pagamento_automatico,
})

,n,churn real (n),churn real (%),% sinalizado,recall,precision,FPR,ROC-AUC
grupo,,,,,,,,
Feminino,709,191,26.94,41.18,0.79,0.51,0.27,0.84
Masculino,700,183,26.14,40.14,0.80,0.52,0.26,0.85
Não idoso,1177,281,23.87,35.43,0.75,0.50,0.23,0.84
Idoso,232,93,40.09,67.24,0.94,0.56,0.50,0.81
Sem parceiro(a),689,231,33.53,50.36,0.81,0.54,0.35,0.82
Com parceiro(a),720,143,19.86,31.39,0.76,0.48,0.20,0.85
Sem dependentes,971,309,31.82,48.92,0.83,0.54,0.33,0.83
Com dependentes,438,65,14.84,22.37,0.63,0.42,0.15,0.82
Pagamento: cheque eletrônico,461,209,45.34,72.02,0.94,0.59,0.54,0.81


## Impacto de negócio (seção 5.6 e ML Canvas)

Traduz o desempenho do modelo em clientes e receita. A receita em risco de um cliente é a sua `MonthlyCharges`,
na moeda do dataset (dólares), considerando **um único mês** de mensalidade.

**Taxa de churn e receita em risco:** os clientes que cancelam pagam, em média, mais do que os que ficam, então a
perda de receita é proporcionalmente maior do que a perda de clientes.

In [11]:
mensalidade = df_model['MonthlyCharges']

pd.Series({
    'Clientes na base': len(df_model),
    'Taxa de churn (%)': 100 * y.mean(),
    'Mensalidade média de quem cancela': mensalidade[y == 1].mean(),
    'Mensalidade média de quem fica': mensalidade[y == 0].mean(),
    'Receita mensal da base': mensalidade.sum(),
    'Receita mensal perdida com churn': mensalidade[y == 1].sum(),
    'Receita perdida / receita total (%)': 100 * mensalidade[y == 1].sum() / mensalidade.sum(),
}).round(2).to_frame('valor')

,valor
Clientes na base,7043.00
Taxa de churn (%),26.54
Mensalidade média de quem cancela,74.44
Mensalidade média de quem fica,61.27
Receita mensal da base,456116.60
Receita mensal perdida com churn,139130.85
Receita perdida / receita total (%),30.50


**Receita em risco detectada por limiar.** O *lift* compara a taxa de churn entre os clientes sinalizados com a
taxa de churn da base: lift 2 significa que uma lista feita pelo modelo tem o dobro de clientes em risco que uma
lista aleatória do mesmo tamanho.

In [12]:
mensalidade_teste = X_test['MonthlyCharges']
receita_em_risco = mensalidade_teste[y_test == 1].sum()

linhas = []
for limiar in [0.3, 0.4, 0.5, 0.6, 0.7]:
    sinalizado = prob >= limiar
    detectado = sinalizado & (y_test == 1)
    linhas.append({
        'limiar': limiar,
        'clientes sinalizados': sinalizado.sum(),
        'churners detectados': detectado.sum(),
        'receita detectada': mensalidade_teste[detectado].sum(),
        '% da receita em risco': 100 * mensalidade_teste[detectado].sum() / receita_em_risco,
        'lift': y_test[sinalizado].mean() / y_test.mean(),
    })

print(f'Receita mensal em risco no teste: {receita_em_risco:.0f} ({(y_test == 1).sum()} clientes que cancelam)')
pd.DataFrame(linhas).set_index('limiar').round(2)

Receita mensal em risco no teste: 27747 (374 clientes que cancelam)


,clientes sinalizados,churners detectados,receita detectada,% da receita em risco,lift
limiar,,,,,
0.3,818,347,25952.10,93.53,1.60
0.4,700,320,24514.45,88.35,1.72
0.5,573,297,23124.80,83.34,1.95
0.6,475,268,20990.75,75.65,2.13
0.7,349,220,17708.70,63.82,2.37


**Priorização pelos clientes de maior risco.** Se o time de retenção só consegue atender parte da base, qual fração
dos cancelamentos e da receita em risco está no topo da lista ordenada pelo modelo?

In [13]:
linhas = []
for fracao in [0.1, 0.2, 0.3]:
    topo = prob.sort_values(ascending=False).index[:int(len(prob) * fracao)]
    cancela = y_test[topo] == 1
    linhas.append({
        'fração da base contatada': fracao,
        'clientes contatados': len(topo),
        '% dos cancelamentos capturados': 100 * cancela.sum() / (y_test == 1).sum(),
        '% da receita em risco capturada': 100 * mensalidade_teste[topo][cancela].sum() / receita_em_risco,
        'lift': y_test[topo].mean() / y_test.mean(),
    })

pd.DataFrame(linhas).set_index('fração da base contatada').round(2)

,clientes contatados,% dos cancelamentos capturados,% da receita em risco capturada,lift
fração da base contatada,,,,
0.1,140,28.61,31.75,2.88
0.2,281,52.41,57.46,2.63
0.3,422,66.84,71.48,2.23


**Simulação de retorno (premissas hipotéticas).** O dataset não informa o custo de uma ação de retenção nem a
chance de ela funcionar. Os valores abaixo são **premissas ilustrativas, a serem substituídas pelos valores reais da
operadora**, e servem para mostrar como comparar os cenários:

`valor líquido = taxa de sucesso × receita detectada − custo por contato × clientes contatados`

In [14]:
TAXA_SUCESSO = 0.20     # hipótese: 20% dos clientes em risco abordados desistem de cancelar
CUSTO_POR_CONTATO = 5   # hipótese: custo da ação por cliente abordado

cenarios = {'Contatar toda a base': pd.Series(True, index=prob.index)}
for limiar in [0.3, 0.4, 0.5, 0.6, 0.7]:
    cenarios[f'Modelo, limiar {limiar}'] = prob >= limiar

linhas = []
for nome, sinalizado in cenarios.items():
    receita = mensalidade_teste[sinalizado & (y_test == 1)].sum()
    linhas.append({
        'cenário': nome,
        'clientes contatados': sinalizado.sum(),
        'receita preservada': TAXA_SUCESSO * receita,
        'custo': CUSTO_POR_CONTATO * sinalizado.sum(),
        'valor líquido': TAXA_SUCESSO * receita - CUSTO_POR_CONTATO * sinalizado.sum(),
    })

pd.DataFrame(linhas).set_index('cenário').round(0)

,clientes contatados,receita preservada,custo,valor líquido
cenário,,,,
Contatar toda a base,1409,5549.0,7045,-1496.0
"Modelo, limiar 0.3",818,5190.0,4090,1100.0
"Modelo, limiar 0.4",700,4903.0,3500,1403.0
"Modelo, limiar 0.5",573,4625.0,2865,1760.0
"Modelo, limiar 0.6",475,4198.0,2375,1823.0
"Modelo, limiar 0.7",349,3542.0,1745,1797.0


In [15]:
# Custo máximo por contato para a ação se pagar no limiar padrão, para diferentes taxas de sucesso
sinalizado = prob >= LIMIAR_CHURN
receita_detectada = mensalidade_teste[sinalizado & (y_test == 1)].sum()

pd.DataFrame({
    'taxa de sucesso': taxa,
    'custo máximo por contato': round(taxa * receita_detectada / sinalizado.sum(), 2),
} for taxa in [0.1, 0.2, 0.3]).set_index('taxa de sucesso')

,custo máximo por contato
taxa de sucesso,
0.1,4.04
0.2,8.07
0.3,12.11
